# Bias stack


bierze zdefiniowane noce, znajduje zera, sprawdza czy sa spelniaja header i robi ich liste. 
potem dla kazdego - robi segment i dla kazdego fitsa z tego segmentu robi statystyki a na koniec laczy ze soba segmenty. 

wyniki zapiuje w `/data/fits/<tel>/raw/<night>/`


In [1]:
import os
import glob
import numpy as np
import pandas as pd
from astropy.io import fits
from tqdm.notebook import tqdm

TEL = 'wk06'
N_START = 1300    # first (oldest) night, int(JD) - 2460000
N_STOP = 1310     # last (newest) night

RAW_DIR = f'/data/fits/{TEL}/raw'
OUT_DIR = os.path.expanduser(f'~/data/ocm_bias/{TEL}')


## Find the zeros

In [2]:

f_list = []
for night in tqdm(range(N_START, N_STOP + 1), desc='nights'):
    files = sorted(glob.glob(f'{RAW_DIR}/{night}/*.fits'))
    for f in tqdm(files, desc=f'night {night}', leave=False):
        h = fits.getheader(f)
        if h['IMAGETYP'] == 'zero':
            f_list.append(f)


nights:   0%|          | 0/11 [00:00<?, ?it/s]

night 1300:   0%|          | 0/72 [00:00<?, ?it/s]

night 1301: 0it [00:00, ?it/s]

night 1302:   0%|          | 0/73 [00:00<?, ?it/s]

night 1303:   0%|          | 0/165 [00:00<?, ?it/s]

night 1304:   0%|          | 0/355 [00:00<?, ?it/s]

... (3 outputs left out)

night 1308:   0%|          | 0/418 [00:00<?, ?it/s]

night 1309:   0%|          | 0/439 [00:00<?, ?it/s]

night 1310:   0%|          | 0/485 [00:00<?, ?it/s]

['/data/fits/wk06/raw/1300/wk06c_1300_48734.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48742.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48750.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48758.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48766.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48774.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48783.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48791.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48800.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48809.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48818.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48782.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48793.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48802.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48813.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48821.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48836.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48844.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48854.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48863.fits',

## Check the headers

A zero must have `EXPTIME = 0`, the CCD at its set temperature, and the same camera mode as the others.

In [3]:
zero_list = []
for f in f_list:
    if float(h['T-CAM']) <  -59:
        zero_list.append(f)
    else:
        ptiny(f,h['T-CAM'])


['/data/fits/wk06/raw/1300/wk06c_1300_48734.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48742.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48750.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48758.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48766.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48774.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48783.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48791.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48800.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48809.fits', '/data/fits/wk06/raw/1300/wk06c_1300_48818.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48782.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48793.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48802.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48813.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48821.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48836.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48844.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48854.fits', '/data/fits/wk06/raw/1302/wk06c_1302_48863.fits',

## Stack

All zeros together would not fit in memory (100 zeros of 2048 x 2048 in float32 = 1.6 GB, more for bigger cameras),
so the image is cut into horizontal bands of `SEG` rows. For one band the same rows are read from every zero, which
gives a small cube `(number of zeros, SEG, 2048)`; the statistics are computed along the first axis (over zeros),
i.e. for every pixel separately, and written into the matching rows of the full maps. Then the next band.

In [4]:
from astropy.stats import sigma_clip

SEG = 256  # wysokosc jednego paska
SIGMA = 5  # mean and std without values further than SIGMA std_mad from the pixel median (cosmics)

# rozmiar obrazka, brany z headera pierwszego na liscie
h = fits.getheader(zero_list[0])
ny, nx = h['NAXIS2'], h['NAXIS1']

# robimy pelna tablice
mean = np.zeros((ny, nx), np.float32)
median = np.zeros((ny, nx), np.float32)
std = np.zeros((ny, nx), np.float32)
std_mad = np.zeros((ny, nx), np.float32)

for y0 in tqdm(range(0, ny, SEG), desc='bands'):  # robimy liste do ny, cos szerokosc paska
    y1 = min(y0 + SEG, ny)    # koniec paska

    # read rows y0..y1 from every zero; .section reads only this part of the file, not the whole image
    paski = []
    for f in tqdm(zero_list, desc=f'rows {y0}', leave=False):
        with fits.open(f) as hdul:
            paski.append(hdul[0].section[y0:y1, :].astype(np.float32))
    paski = np.array(paski)  

    # axis=0 = statystyki w osi 0
    # sigma_clip masks, for every pixel separately, the values further than SIGMA sigma from its median (repeated
    # until nothing more is masked); mean and std are then computed from the values left. The sigma is from the MAD
    # (stdfunc='mad_std'), not the plain std: a cosmic inflates the std itself, so with few zeros it could never be
    # 5 std away (one value among N is at most (N-1)/sqrt(N) std from the rest: 2.8 for N = 10)
    clipped = sigma_clip(paski, sigma=SIGMA, axis=0, stdfunc='mad_std')
    mean[y0:y1] = clipped.mean(axis=0)
    std[y0:y1] = clipped.std(axis=0)
    median[y0:y1] = np.median(paski, axis=0)
    # MAD x 1.4826 = the std equivalent of the MAD (equal to std for Gaussian noise)
    std_mad[y0:y1] = 1.4826 * np.median(np.abs(paski - median[y0:y1]), axis=0)


bands:   0%|          | 0/8 [00:00<?, ?it/s]

rows 0:   0%|          | 0/209 [00:00<?, ?it/s]

rows 256:   0%|          | 0/209 [00:00<?, ?it/s]

rows 512:   0%|          | 0/209 [00:00<?, ?it/s]

rows 768:   0%|          | 0/209 [00:00<?, ?it/s]

rows 1024:   0%|          | 0/209 [00:00<?, ?it/s]

rows 1280:   0%|          | 0/209 [00:00<?, ?it/s]

rows 1536:   0%|          | 0/209 [00:00<?, ?it/s]

rows 1792:   0%|          | 0/209 [00:00<?, ?it/s]

In [5]:
# common header: what the maps were made from
hdr = fits.Header()
hdr['TELESCOP'] = TEL
hdr['NIGHT-BG'] = (N_START, 'First night')
hdr['NIGHT-EN'] = (N_STOP, 'Last night')
hdr['NZERO'] = (len(zero_list), 'Number of zeros in the stack')
hdr['SIGCLIP'] = (SIGMA, 'Sigma clipping (MAD sigma) of mean and std')
for i, f in enumerate(zero_list):
    hdr[f'FILE{i + 1:03d}'] = os.path.basename(f)

# one FITS file per map, e.g. ~/data/ocm_bias/wk06/wk06_1340_1344_mean.fits
os.makedirs(OUT_DIR, exist_ok=True)
for name, m in [('mean', mean), ('median', median), ('std', std), ('std_mad', std_mad)]:
    out = f'{OUT_DIR}/{TEL}_{N_START}_{N_STOP}_{name}.fits'
    fits.writeto(out, m, hdr, overwrite=True)
    print(out)

/home/mgorski/data/ocm_bias/wk06/wk06_1300_1310_mean.fits
/home/mgorski/data/ocm_bias/wk06/wk06_1300_1310_median.fits
/home/mgorski/data/ocm_bias/wk06/wk06_1300_1310_std.fits
/home/mgorski/data/ocm_bias/wk06/wk06_1300_1310_std_mad.fits


## What else can be done

What the stacks give: long-term trends of the detector and pixels with problems. Ideas, not done yet:

**Trends in time**
- Stacks week by week (or month by month) and each compared with one reference stack: difference of the mean maps
  (level), ratio of the std maps (noise), for the whole frame, per block and per pixel.
- Global numbers per stack against time: level, noise, number of flagged pixels; against `T-CAM`, dusk vs dawn series,
  dome / mirror cover open or closed (`DOME-SHT`, `MIRR-COV`, about 1/3 of the zeros are taken with the dome open).
- Noise vs the header: `RON = 10.4 e-` at `GAIN = 0.97` is ~10.7 ADU, the measured std is ~6.7-7 ADU.

**Pixels with problems**
- Pixels above / below 5 sigma: are they the same in every stack (a defect) or different each time (noise)?
- Hot / cold columns and rows: medians of columns and rows of the mean map.
- Telegraph (RTS) pixels: high `std / std_mad`, two-level histogram of the pixel values over the zeros.
- How many values sigma clipping rejected per pixel (a map of the cosmic hits; the edge rows?).

**Structure of the frame**
- The ~256 px block pattern in the mean map and the brighter bottom rows (y < ~100): stable or changing?
- Large-scale gradient and background structure with `pyaraucaria.ffs` (branch `mgorski_dev`, not installed on this
  machine yet), on the mean map and on single zeros.

**OFP**
- OFP keeps every zero series in `/data/fits/<tel>/processed-ofp/zeros/<series>/`: symlinks to the raw zeros, the
  master `<series>_master_z.fits` and a json with the full header and basic stats (mean, median, std) of every frame.
  Listing these directories finds the zeros without opening any file (the header scan takes ~0.17 s per file).
- Compare our stack with the OFP masters; the OFP json stats give a time series of single zeros for free.

**Next**
- Single zeros, each against a reference stack (separate notebook): level offset, noise, residual structure, cosmics.
- Darks: the same approach.
- Other telescopes (zb08, jk15): several readout modes (OFP keeps `last_zero_r<READ-MOD>g<GAIN-MOD>.fits`), so stacks
  must be split by `READ-MOD` / `GAIN-MOD`.